# GAT-GRU v01 — node-feature ablation

Giữ nguyên kiến trúc, edge features, loss và early-stopping objective của baseline; chỉ thay node-feature set. Ranking được macro-average theo run/scenario. Chỉ dùng train/validation.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / 'modeling-common').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'modeling-common'))
import forecast_ablation as fa

ctx = fa.load_context(ROOT)
cfg = fa.load_baseline_config(ctx)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SCREENING_SEEDS = (0,)  # broad screen; finalists below use five paired seeds
OUT = ROOT / 'artifacts' / 'gatgru' / 'v01_node_ablation'
print('device:', DEVICE, '| screening seeds:', SCREENING_SEEDS, '| config:', cfg)

device: cpu | screening seeds: (0,) | config: {'window': 18, 'dropout': 0.055739332533562704, 'lr': 0.0015458145265688748, 'batch_size': 64, 'gat_hidden': 32, 'heads': 4, 'gat_layers': 2, 'gru_hidden': 128, 'edge_direction': 'both', 'use_edge_features': True}


## A. Group ablation

In [2]:
group_summary = fa.run_fixed_variants(
    ctx, 'v01_node_group_ablation', OUT / 'group',
    fa.node_group_variants(cfg), seeds=SCREENING_SEEDS, device=DEVICE,
    score_weights=fa.BASELINE_SCORE_WEIGHTS, parent_version='baseline-2026-10-14',
)
display(group_summary)

,variant,model_kind,node_features,edge_features,edge_control,topology_control,n_seeds,primary_score_mean,primary_score_std,cpu_weighted_diagnostic_mean,...,cpu_cores_mae_macro_mean,cpu_cores_mae_macro_std,train_seconds_mean,train_seconds_std,early_stop_val_mae_mean,early_stop_val_mae_std,n_params_mean,n_params_std,epochs_trained_mean,epochs_trained_std
8,full_minus_cpu_resource,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,1,0.114665,0.0,0.107230,...,0.013743,0.0,595.439285,0.0,0.114641,0.0,121560.0,0.0,30.0,0.0
3,core_plus_latency,gatgru,"[rps_in, latency_p50_ms, latency_p95_ms, cpu_c...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,1,0.115011,0.0,0.107161,...,0.013540,0.0,614.512291,0.0,0.114987,0.0,120280.0,0.0,32.0,0.0
1,core_plus_load,gatgru,"[rps_in, rps_in_delta, rps_per_replica, cpu_co...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,1,0.115148,0.0,0.108077,...,0.013963,0.0,951.866848,0.0,0.115130,0.0,120280.0,0.0,31.0,0.0
7,full_minus_load,gatgru,"[rps_in, latency_p50_ms, latency_p95_ms, cpu_c...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,1,0.115215,0.0,0.107370,...,0.013555,0.0,531.278191,0.0,0.115195,0.0,121816.0,0.0,27.0,0.0
2,core_plus_cpu,gatgru,"[rps_in, cpu_cores, cpu_util_request, cpu_thro...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,1,0.115561,0.0,0.108142,...,0.013860,0.0,903.561599,0.0,0.115546,0.0,120536.0,0.0,32.0,0.0
9,full_minus_latency,gatgru,"[rps_in, rps_in_delta, rps_per_replica, cpu_co...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,1,0.115579,0.0,0.108192,...,0.013864,0.0,536.113053,0.0,0.115564,0.0,121816.0,0.0,27.0,0.0
4,core_load_cpu,gatgru,"[rps_in, rps_in_delta, rps_per_replica, cpu_co...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,1,0.115820,0.0,0.108706,...,0.014007,0.0,506.473638,0.0,0.115803,0.0,121048.0,0.0,30.0,0.0
5,core_load_cpu_latency,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,1,0.116275,0.0,0.108641,...,0.013830,0.0,517.225908,0.0,0.116253,0.0,121560.0,0.0,30.0,0.0
10,full_minus_memory_network,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,1,0.116275,0.0,0.108641,...,0.013830,0.0,589.540260,0.0,0.116253,0.0,121560.0,0.0,30.0,0.0
6,full13,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,1,0.116475,0.0,0.108728,...,0.013848,0.0,585.858322,0.0,0.116451,0.0,122328.0,0.0,30.0,0.0


## B. Leave-one-feature-out

Đây là screening độc lập quanh full-13. Có thể chạy tiếp cell này sau group ablation; artifacts theo seed có thể resume.

In [3]:
loo_summary = fa.run_fixed_variants(
    ctx, 'v01_node_leave_one_out', OUT / 'leave_one_out',
    fa.node_leave_one_out_variants(cfg), seeds=SCREENING_SEEDS, device=DEVICE,
    score_weights=fa.BASELINE_SCORE_WEIGHTS, parent_version='baseline-2026-10-14',
)
display(loo_summary)

,variant,model_kind,node_features,edge_features,edge_control,topology_control,n_seeds,primary_score_mean,primary_score_std,cpu_weighted_diagnostic_mean,...,cpu_cores_mae_macro_mean,cpu_cores_mae_macro_std,train_seconds_mean,train_seconds_std,early_stop_val_mae_mean,early_stop_val_mae_std,n_params_mean,n_params_std,epochs_trained_mean,epochs_trained_std
6,without_cpu_sidecar_cores,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,1,0.114541,0.0,0.106830,...,0.013562,0.0,572.939025,0.0,0.114520,0.0,122072.0,0.0,27.0,0.0
5,without_cpu_throttle_ratio,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,1,0.114653,0.0,0.107250,...,0.013728,0.0,552.446612,0.0,0.114633,0.0,122072.0,0.0,27.0,0.0
1,without_rps_per_replica,gatgru,"[rps_in, rps_in_delta, latency_p50_ms, latency...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,1,0.114715,0.0,0.107042,...,0.013617,0.0,663.456730,0.0,0.114694,0.0,122072.0,0.0,35.0,0.0
3,without_latency_p95_ms,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,1,0.114881,0.0,0.107373,...,0.013702,0.0,551.213582,0.0,0.114861,0.0,122072.0,0.0,27.0,0.0
0,without_rps_in_delta,gatgru,"[rps_in, rps_per_replica, latency_p50_ms, late...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,1,0.114915,0.0,0.106966,...,0.013439,0.0,622.607544,0.0,0.114892,0.0,122072.0,0.0,32.0,0.0
7,without_mem_mib,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,1,0.114964,0.0,0.107240,...,0.013625,0.0,564.119319,0.0,0.114945,0.0,122072.0,0.0,27.0,0.0
8,without_net_rx_kBps,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,1,0.115171,0.0,0.107751,...,0.013774,0.0,663.296466,0.0,0.115151,0.0,122072.0,0.0,32.0,0.0
9,without_net_tx_kBps,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,1,0.115477,0.0,0.107746,...,0.013742,0.0,386.034453,0.0,0.115455,0.0,122072.0,0.0,19.0,0.0
2,without_latency_p50_ms,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,1,0.115730,0.0,0.108233,...,0.013833,0.0,520.348175,0.0,0.115712,0.0,122072.0,0.0,27.0,0.0
4,without_cpu_util_request,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,1,0.116629,0.0,0.109100,...,0.013923,0.0,640.205295,0.0,0.116606,0.0,122072.0,0.0,32.0,0.0


## C. Ranking for review

Không tự động promote feature. Sau khi chạy, review `candidate_ranking.csv` rồi cập nhật `gatgru_feature_contract_v01.json`.

In [4]:
ranking = pd.concat([group_summary.assign(stage='group'), loo_summary.assign(stage='leave_one_out')], ignore_index=True)
ranking = ranking.sort_values(['primary_score_mean', 'n_params_mean'])
ranking.to_csv(OUT / 'candidate_ranking.csv', index=False)
display(ranking[['stage', 'variant', 'primary_score_mean', 'primary_score_std',
                 'cpu_weighted_diagnostic_mean', 'rps_in_mae_macro_mean',
                 'cpu_cores_mae_macro_mean', 'n_params_mean', 'train_seconds_mean']])

,stage,variant,primary_score_mean,primary_score_std,cpu_weighted_diagnostic_mean,rps_in_mae_macro_mean,cpu_cores_mae_macro_mean,n_params_mean,train_seconds_mean
11,leave_one_out,without_cpu_sidecar_cores,0.114541,0.0,0.106830,1.018308,0.013562,122072.0,572.939025
12,leave_one_out,without_cpu_throttle_ratio,0.114653,0.0,0.107250,1.015903,0.013728,122072.0,552.446612
0,group,full_minus_cpu_resource,0.114665,0.0,0.107230,1.027653,0.013743,121560.0,595.439285
13,leave_one_out,without_rps_per_replica,0.114715,0.0,0.107042,1.040595,0.013617,122072.0,663.456730
14,leave_one_out,without_latency_p95_ms,0.114881,0.0,0.107373,1.011985,0.013702,122072.0,551.213582
15,leave_one_out,without_rps_in_delta,0.114915,0.0,0.106966,1.022247,0.013439,122072.0,622.607544
16,leave_one_out,without_mem_mib,0.114964,0.0,0.107240,1.020290,0.013625,122072.0,564.119319
1,group,core_plus_latency,0.115011,0.0,0.107161,1.017842,0.013540,120280.0,614.512291
2,group,core_plus_load,0.115148,0.0,0.108077,1.013050,0.013963,120280.0,951.866848
17,leave_one_out,without_net_rx_kBps,0.115171,0.0,0.107751,1.016533,0.013774,122072.0,663.296466


## D. Five-seed confirmation

Xác nhận bốn candidate đứng đầu và luôn giữ `full13` làm anchor. Kết quả này mới là bằng chứng chính để duyệt node-feature contract.

In [5]:
top = ranking.drop_duplicates('variant').head(4)
anchor = ranking[ranking.variant == 'full13']
finalists = pd.concat([top, anchor], ignore_index=True).drop_duplicates('variant')
finalist_specs = [
    dict(name='confirm_' + row.variant, kind='gatgru', config=cfg,
         node_features=list(row.node_features), edge_features=fa.EDGE_FEATURES)
    for row in finalists.itertuples()
]
confirmation = fa.run_fixed_variants(
    ctx, 'v01_node_finalist_confirmation', OUT / 'finalists', finalist_specs,
    seeds=(0, 1, 2, 3, 4), device=DEVICE, score_weights=fa.BASELINE_SCORE_WEIGHTS,
    parent_version='baseline-2026-10-14',
)
display(confirmation)

,variant,model_kind,node_features,edge_features,edge_control,topology_control,n_seeds,primary_score_mean,primary_score_std,cpu_weighted_diagnostic_mean,...,cpu_cores_mae_macro_mean,cpu_cores_mae_macro_std,train_seconds_mean,train_seconds_std,early_stop_val_mae_mean,early_stop_val_mae_std,n_params_mean,n_params_std,epochs_trained_mean,epochs_trained_std
2,confirm_full_minus_cpu_resource,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,5,0.113994,0.001107,0.106535,...,0.013603,0.000213,749.823635,88.259382,0.113970,0.001106,121560.0,0.0,27.4,1.816590
1,confirm_without_cpu_throttle_ratio,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,5,0.114723,0.000585,0.107220,...,0.013667,0.000083,559.654798,23.879603,0.114701,0.000586,122072.0,0.0,25.6,2.190890
3,confirm_without_rps_per_replica,gatgru,"[rps_in, rps_in_delta, latency_p50_ms, latency...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,5,0.115010,0.000483,0.107293,...,0.013635,0.000135,975.645538,488.905785,0.114988,0.000482,122072.0,0.0,27.6,5.458938
0,confirm_without_cpu_sidecar_cores,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,5,0.115405,0.000889,0.107854,...,0.013752,0.000215,565.092352,34.862751,0.115384,0.000890,122072.0,0.0,26.6,1.673320
4,confirm_full13,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,5,0.115510,0.001158,0.107878,...,0.013750,0.000227,499.404651,119.796578,0.115486,0.001157,122328.0,0.0,25.8,2.949576
